# Upload the 16 CC-Perturb-seq analysis files to IGVF

This notebook uploads only the **16 newly prepared analysis files**:

- 4 day-specific annotated `.h5ad` MatrixFiles
- 4 day-specific PerturbNMF `.h5mu` MatrixFiles
- 4 `guide_quantifications.tsv.gz` TabularFiles
- 4 `per_cell_quality_report.tsv.gz` TabularFiles

It does **not** recreate the previously uploaded gene-program, gene-universe, regulator, or correlation files, and it does not upload the removed cis/trans differential-effect files.

The submission is deliberately split into dependency-safe phases:

1. local integrity and metadata preflight;
2. dry-run the four h5ad files;
3. explicitly authorize and submit the four h5ad files;
4. verify the h5ad Portal records;
5. dry-run the four h5mu and eight tabular files;
6. explicitly authorize and submit those 12 dependent files;
7. reconcile all 16 Portal records and export accessions and links.

The real upload cells are blocked by exact confirmation tokens. Re-running the notebook is safe: it queries each alias first and skips a record only when the existing Portal record matches the expected MD5 and AnalysisSet. It stops on any mismatch or ambiguous partial submission.

Run this notebook on a Sherlock compute node with the existing `igvf-upload-ready` kernel. The notebook calls `iu_register.py` from Tri's separate `igvf_utils` environment.

In [1]:
from pathlib import Path

# Final files and metadata generated by 01_make_igvf_upload_ready_files.ipynb.
OUTPUT_ROOT = Path(
    "/oak/stanford/groups/engreitz/Projects/IGVF/"
    "CC-Perturb-seq/300gene/analysis_submission"
)
METADATA_DIR = OUTPUT_ROOT / "metadata"
LOG_DIR = OUTPUT_ROOT / "logs" / "igvf_utils_analysis_upload"
UPLOAD_WORK_DIR = OUTPUT_ROOT / "_upload_work"
ONE_ROW_DIR = UPLOAD_WORK_DIR / "one_row_tsv"

READY_MARKER = OUTPUT_ROOT / "READY_TO_UPLOAD_16_FILES.txt"
MASTER_MANIFEST = METADATA_DIR / "upload_manifest_16_files.tsv"
H5AD_TSV = METADATA_DIR / "01_matrix_file_h5ad.tsv"
H5MU_TSV = METADATA_DIR / "02_matrix_file_h5mu.tsv"
TABULAR_TSV = METADATA_DIR / "03_tabular_file_step1.tsv"

IGVF_PYTHON = Path(
    "/oak/stanford/groups/engreitz/Users/tri/envs/igvf_utils/bin/python"
)
IU_REGISTER = Path(
    "/oak/stanford/groups/engreitz/Users/tri/envs/igvf_utils/igvf_utils/"
    "igvf_utils/MetaDataRegistration/iu_register.py"
)

IGVF_MODE = "prod"
IGVF_API_BASE = "https://api.data.igvf.org"
IGVF_PORTAL_BASE = "https://data.igvf.org"

ANALYSIS_SETS = {
    "D0": "IGVFDS0463XTUH",
    "D1": "IGVFDS0991DBQZ",
    "D2": "IGVFDS4402ZGWJ",
    "D3": "IGVFDS2307REVJ",
}

# Definitive but potentially time-consuming. Leave True for the first upload.
VERIFY_MD5_BEFORE_UPLOAD = True

# The real upload cells remain blocked until these values are edited exactly.
H5AD_REAL_CONFIRMATION = ""
DEPENDENT_REAL_CONFIRMATION = ""
EXPECTED_H5AD_TOKEN = "UPLOAD_4_H5AD_FILES_TO_IGVF_PROD"
EXPECTED_DEPENDENT_TOKEN = "UPLOAD_12_DEPENDENT_FILES_TO_IGVF_PROD"

# Small pause between real submissions reduces API bursts.
SECONDS_BETWEEN_REAL_UPLOADS = 2.0

for directory in (LOG_DIR, UPLOAD_WORK_DIR, ONE_ROW_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("READY_MARKER:", READY_MARKER)
print("IGVF_PYTHON:", IGVF_PYTHON)
print("IU_REGISTER:", IU_REGISTER)
print("VERIFY_MD5_BEFORE_UPLOAD:", VERIFY_MD5_BEFORE_UPLOAD)
print("H5AD real upload authorized:", H5AD_REAL_CONFIRMATION == EXPECTED_H5AD_TOKEN)
print("Dependent real upload authorized:", DEPENDENT_REAL_CONFIRMATION == EXPECTED_DEPENDENT_TOKEN)

In [2]:
import os
from getpass import getpass


def enter_igvf_credentials() -> None:
    """Prompt securely for IGVF production credentials."""

    api_key = getpass("IGVF API key: ").strip()
    secret_key = getpass("IGVF secret key: ").strip()

    if not api_key:
        raise ValueError("IGVF API key was empty.")

    if not secret_key:
        raise ValueError("IGVF secret key was empty.")

    # Make the credentials available to Python and subprocesses such as
    # iu_register.py launched by subprocess.run().
    os.environ["IGVF_API_KEY"] = api_key
    os.environ["IGVF_SECRET_KEY"] = secret_key

    # Remove the temporary local references.
    del api_key
    del secret_key

    print("IGVF credentials are set for the current notebook kernel.")


enter_igvf_credentials()

## Helper functions

These functions use plain-text output only. They do not depend on the VS Code notebook renderer.

In [3]:
import base64
import csv
import getpass
import hashlib
import json
import os
import re
import subprocess
import time
from datetime import datetime, timezone
from typing import Any
from urllib.error import HTTPError, URLError
from urllib.parse import quote
from urllib.request import Request, urlopen

import pandas as pd

ERROR_PATTERNS = re.compile(
    r"Traceback|ValidationFailure|Failed to POST|Unprocessable|Forbidden|"
    r"Unauthorized|Permission denied|401 Client Error|403 Client Error|"
    r"409 Client Error|KeyError|JSONDecodeError|ReadTimeout|ConnectionError",
    flags=re.IGNORECASE,
)
ACCESSION_PATTERN = re.compile(r"IGVFFI[A-Z0-9]+")


def now_utc() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def is_blank(value: Any) -> bool:
    if value is None:
        return True
    if isinstance(value, (str, bytes)):
        return not str(value).strip()
    try:
        return bool(pd.isna(value))
    except (TypeError, ValueError):
        return False


def normalize_identifier(value: Any) -> str:
    if isinstance(value, dict):
        value = value.get("@id") or value.get("accession") or ""
    if is_blank(value):
        return ""
    text = str(value).strip()
    if not text:
        return ""
    if text.startswith("http://") or text.startswith("https://"):
        text = text.rstrip("/").split("/")[-1]
    elif text.startswith("/"):
        text = text.rstrip("/").split("/")[-1]
    return text


def normalize_aliases(value: Any) -> list[str]:
    if isinstance(value, list):
        return [str(item) for item in value]
    if isinstance(value, str) and value:
        return [value]
    return []


def split_identifiers(value: Any) -> list[str]:
    if is_blank(value):
        return []
    return [item.strip() for item in str(value).split(",") if item.strip()]


def credential_headers() -> dict[str, str]:
    key = os.environ.get("IGVF_API_KEY", "")
    secret = os.environ.get("IGVF_SECRET_KEY", "")
    if not key or not secret:
        raise RuntimeError("IGVF_API_KEY and IGVF_SECRET_KEY are not set in this kernel")
    encoded = base64.b64encode(f"{key}:{secret}".encode()).decode()
    return {
        "Authorization": f"Basic {encoded}",
        "Accept": "application/json",
        "User-Agent": "cc-perturbseq-analysis-upload-notebook/1.0",
    }


def portal_get(
    identifier: str,
    *,
    attempts: int = 5,
    wait_seconds: float = 2.0,
) -> dict[str, Any] | None:
    ident = normalize_identifier(identifier)
    if not ident:
        raise ValueError("Cannot query an empty Portal identifier")
    encoded_ident = quote(ident, safe=":@._-")
    url = (
        f"{IGVF_API_BASE}/{encoded_ident}/"
        "?format=json&datastore=database&frame=object"
    )
    last_error: Exception | None = None
    for attempt in range(1, attempts + 1):
        request = Request(url, headers=credential_headers(), method="GET")
        try:
            with urlopen(request, timeout=60) as response:
                return json.loads(response.read().decode("utf-8"))
        except HTTPError as exc:
            if exc.code == 404:
                return None
            if exc.code in {401, 403}:
                body = exc.read().decode("utf-8", errors="replace")[:500]
                raise RuntimeError(
                    f"Portal authentication/permission failure for {ident}: "
                    f"HTTP {exc.code}: {body}"
                ) from exc
            last_error = exc
            if exc.code not in {429, 500, 502, 503, 504} or attempt == attempts:
                body = exc.read().decode("utf-8", errors="replace")[:500]
                raise RuntimeError(
                    f"Portal GET failed for {ident}: HTTP {exc.code}: {body}"
                ) from exc
        except (URLError, TimeoutError, json.JSONDecodeError) as exc:
            last_error = exc
            if attempt == attempts:
                raise RuntimeError(f"Portal GET failed for {ident}: {exc}") from exc
        time.sleep(wait_seconds * attempt)
    raise RuntimeError(f"Portal GET failed for {ident}: {last_error}")


def portal_url(accession: str) -> str:
    return f"{IGVF_PORTAL_BASE}/{accession}/" if accession else ""


def record_comparison(row: dict[str, str], obj: dict[str, Any]) -> list[str]:
    issues: list[str] = []
    expected_alias = row["aliases"]
    aliases = normalize_aliases(obj.get("aliases"))
    if expected_alias not in aliases:
        issues.append(f"alias missing on Portal: {expected_alias}")

    expected_md5 = row["md5sum"]
    portal_md5 = str(obj.get("md5sum") or "")
    if portal_md5 != expected_md5:
        issues.append(f"md5sum mismatch: expected={expected_md5}, portal={portal_md5}")

    expected_file_set = normalize_identifier(row["file_set"])
    portal_file_set = normalize_identifier(obj.get("file_set"))
    if portal_file_set != expected_file_set:
        issues.append(
            f"file_set mismatch: expected={expected_file_set}, portal={portal_file_set}"
        )

    expected_format = row.get("file_format", "")
    portal_format = str(obj.get("file_format") or "")
    if portal_format and portal_format != expected_format:
        issues.append(
            f"file_format mismatch: expected={expected_format}, portal={portal_format}"
        )

    expected_content = row.get("content_type", "")
    portal_content = str(obj.get("content_type") or "")
    if portal_content and portal_content != expected_content:
        issues.append(
            f"content_type mismatch: expected={expected_content}, portal={portal_content}"
        )

    upload_status = str(obj.get("upload_status") or "").lower()
    if upload_status in {"file not found", "invalidated"}:
        issues.append(f"Portal upload_status is {upload_status!r}")
    object_status = str(obj.get("status") or "").lower()
    if object_status in {"revoked", "archived", "deleted"}:
        issues.append(f"Portal object status is {object_status!r}")
    return issues


def wait_for_record(
    alias: str,
    *,
    attempts: int = 12,
    wait_seconds: float = 5.0,
) -> dict[str, Any] | None:
    for attempt in range(1, attempts + 1):
        obj = portal_get(alias, attempts=3)
        if obj:
            return obj
        if attempt < attempts:
            time.sleep(wait_seconds)
    return None


def md5sum(path: Path, block_size: int = 16 * 1024 * 1024) -> str:
    digest = hashlib.md5()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()


def load_metadata(path: Path, profile: str, stage: str) -> pd.DataFrame:
    frame = pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False)
    if frame.empty:
        raise ValueError(f"Metadata file is empty: {path}")
    if any(column.startswith("#") for column in frame.columns):
        raise ValueError(f"Response/helper columns beginning with # found in {path}")
    empty_columns = [column for column in frame.columns if not frame[column].str.strip().any()]
    if empty_columns:
        print(f"Removing all-empty columns from {path.name}: {empty_columns}")
        frame = frame.drop(columns=empty_columns)
    frame["_profile"] = profile
    frame["_stage"] = stage
    frame["_source_tsv"] = str(path)
    return frame


def safe_slug(text: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", text).strip("_")


def write_one_row_tsv(row: dict[str, str], profile: str, index: int) -> Path:
    alias = row["aliases"]
    path = ONE_ROW_DIR / f"{index:02d}_{profile}_{safe_slug(alias)}.tsv"
    payload = {
        key: "" if is_blank(value) else str(value)
        for key, value in row.items()
        if not key.startswith("_")
    }
    fields = [key for key, value in payload.items() if value.strip()]
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=fields,
            delimiter="\t",
            lineterminator="\n",
            extrasaction="ignore",
        )
        writer.writeheader()
        writer.writerow(payload)
    return path


def run_command_streaming(command: list[str], log_path: Path) -> int:
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print("Command:", " ".join(command))
    print("Log:", log_path)
    with log_path.open("w", encoding="utf-8") as log_handle:
        process = subprocess.Popen(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=os.environ.copy(),
        )
        try:
            assert process.stdout is not None
            for line in process.stdout:
                print(line, end="")
                log_handle.write(line)
                log_handle.flush()
        except KeyboardInterrupt:
            process.terminate()
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                process.kill()
            raise
        return process.wait()


def run_iu_register(
    row: dict[str, str],
    *,
    dry_run: bool,
    ordinal: int,
) -> dict[str, Any]:
    profile = row["_profile"]
    alias = row["aliases"]
    row_tsv = write_one_row_tsv(row, profile, ordinal)
    mode_label = "dryrun" if dry_run else "submit"
    log_path = LOG_DIR / (
        f"{ordinal:02d}_{profile}_{safe_slug(alias)}_{mode_label}_"
        f"{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
    )
    command = [
        str(IGVF_PYTHON),
        str(IU_REGISTER),
        "-m",
        IGVF_MODE,
        "-p",
        profile,
    ]
    if dry_run:
        command.append("-d")
    command.extend(["-i", str(row_tsv)])

    started = now_utc()
    return_code = run_command_streaming(command, log_path)
    finished = now_utc()
    text = log_path.read_text(encoding="utf-8", errors="replace")
    result = {
        "alias": alias,
        "profile": profile,
        "stage": row["_stage"],
        "dry_run": dry_run,
        "return_code": return_code,
        "dry_run_marker": "DRY RUN is enabled" in text,
        "error_pattern_found": bool(ERROR_PATTERNS.search(text)),
        "aws_upload_success_message": "AWS upload successful" in text,
        "accessions_in_log": ",".join(sorted(set(ACCESSION_PATTERN.findall(text)))),
        "row_tsv": str(row_tsv),
        "log_path": str(log_path),
        "started_utc": started,
        "finished_utc": finished,
    }
    return result


def local_aws_success_for_alias(alias: str) -> bool:
    pattern = f"*_{safe_slug(alias)}_submit_*.log"
    for log_path in LOG_DIR.glob(pattern):
        try:
            if "AWS upload successful" in log_path.read_text(
                encoding="utf-8", errors="replace"
            ):
                return True
        except OSError:
            continue
    return False


def existing_record_is_safely_reusable(
    row: dict[str, str], obj: dict[str, Any]
) -> tuple[bool, list[str]]:
    issues = record_comparison(row, obj)
    if issues:
        return False, issues
    upload_status = str(obj.get("upload_status") or "").lower()
    if upload_status in {"validated", "validation exempted"}:
        return True, []
    if local_aws_success_for_alias(row["aliases"]):
        return True, []
    if upload_status == "pending":
        return False, [
            "matching Portal record is still pending and no local log confirms "
            "'AWS upload successful'; review before skipping or retrying"
        ]
    return True, []


def dry_run_rows(frame: pd.DataFrame, label: str) -> pd.DataFrame:
    results: list[dict[str, Any]] = []
    for ordinal, row in enumerate(frame.to_dict("records"), start=1):
        alias = row["aliases"]
        existing = portal_get(alias)
        if existing:
            reusable, issues = existing_record_is_safely_reusable(row, existing)
            if not reusable:
                raise RuntimeError(
                    f"{alias} already exists but cannot be safely skipped: "
                    + "; ".join(issues)
                )
            results.append({
                "alias": alias,
                "profile": row["_profile"],
                "stage": row["_stage"],
                "dry_run": True,
                "return_code": 0,
                "dry_run_marker": True,
                "error_pattern_found": False,
                "aws_upload_success_message": False,
                "accessions_in_log": str(existing.get("accession") or ""),
                "row_tsv": "",
                "log_path": "",
                "started_utc": now_utc(),
                "finished_utc": now_utc(),
                "status": "ALREADY_EXISTS_MATCH",
            })
            print(f"SKIP dry run; matching record already exists: {alias}")
            continue

        result = run_iu_register(row, dry_run=True, ordinal=ordinal)
        if (
            result["return_code"] != 0
            or not result["dry_run_marker"]
            or result["error_pattern_found"]
        ):
            result["status"] = "DRY_RUN_FAILED"
            results.append(result)
            output = pd.DataFrame(results)
            output.to_csv(
                UPLOAD_WORK_DIR / f"{label}_dryrun_results.tsv",
                sep="\t",
                index=False,
                lineterminator="\n",
            )
            raise RuntimeError(
                f"Dry run failed for {alias}. Review {result['log_path']}"
            )
        result["status"] = "DRY_RUN_PASS"
        results.append(result)

    output = pd.DataFrame(results)
    output_path = UPLOAD_WORK_DIR / f"{label}_dryrun_results.tsv"
    output.to_csv(output_path, sep="\t", index=False, lineterminator="\n")
    print(f"Dry-run results: {output_path}")
    print(output[["alias", "profile", "status", "log_path"]].to_string(index=False))
    return output


def submit_rows(frame: pd.DataFrame, label: str) -> pd.DataFrame:
    results: list[dict[str, Any]] = []
    for ordinal, row in enumerate(frame.to_dict("records"), start=1):
        alias = row["aliases"]
        existing = portal_get(alias)
        if existing:
            reusable, issues = existing_record_is_safely_reusable(row, existing)
            if not reusable:
                raise RuntimeError(
                    f"{alias} already exists but cannot be safely skipped: "
                    + "; ".join(issues)
                )
            result = {
                "alias": alias,
                "profile": row["_profile"],
                "stage": row["_stage"],
                "status": "ALREADY_EXISTS_MATCH",
                "return_code": 0,
                "accession": str(existing.get("accession") or ""),
                "portal_url": portal_url(str(existing.get("accession") or "")),
                "portal_upload_status": str(existing.get("upload_status") or ""),
                "portal_object_status": str(existing.get("status") or ""),
                "log_path": "",
                "submitted_utc": now_utc(),
            }
            results.append(result)
            print(f"SKIP real upload; matching record already exists: {alias}")
            continue

        raw_result = run_iu_register(row, dry_run=False, ordinal=ordinal)
        obj = wait_for_record(alias)
        if obj is None:
            raw_result["status"] = "FAILED_NOT_FOUND_ON_PORTAL"
            results.append(raw_result)
            pd.DataFrame(results).to_csv(
                UPLOAD_WORK_DIR / f"{label}_submit_results.tsv",
                sep="\t",
                index=False,
                lineterminator="\n",
            )
            raise RuntimeError(
                f"{alias} was not found on the Portal after submission. "
                f"Review {raw_result['log_path']} before retrying."
            )

        issues = record_comparison(row, obj)
        accession = str(obj.get("accession") or "")
        result = {
            **raw_result,
            "accession": accession,
            "portal_url": portal_url(accession),
            "portal_upload_status": str(obj.get("upload_status") or ""),
            "portal_object_status": str(obj.get("status") or ""),
            "portal_comparison_issues": "; ".join(issues),
            "submitted_utc": now_utc(),
        }

        if issues:
            result["status"] = "PORTAL_RECORD_MISMATCH"
            results.append(result)
            pd.DataFrame(results).to_csv(
                UPLOAD_WORK_DIR / f"{label}_submit_results.tsv",
                sep="\t",
                index=False,
                lineterminator="\n",
            )
            raise RuntimeError(
                f"Portal record mismatch for {alias}: " + "; ".join(issues)
            )

        if raw_result["return_code"] != 0:
            result["status"] = "PORTAL_EXISTS_AFTER_CLIENT_ERROR_REVIEW_REQUIRED"
            results.append(result)
            pd.DataFrame(results).to_csv(
                UPLOAD_WORK_DIR / f"{label}_submit_results.tsv",
                sep="\t",
                index=False,
                lineterminator="\n",
            )
            raise RuntimeError(
                f"{alias} exists on the Portal, but iu_register exited nonzero. "
                f"Do not retry blindly; review {raw_result['log_path']} and the Portal record."
            )

        if not raw_result["aws_upload_success_message"]:
            result["status"] = "PORTAL_EXISTS_BUT_AWS_UPLOAD_NOT_CONFIRMED"
            results.append(result)
            pd.DataFrame(results).to_csv(
                UPLOAD_WORK_DIR / f"{label}_submit_results.tsv",
                sep="\t",
                index=False,
                lineterminator="\n",
            )
            raise RuntimeError(
                f"{alias} exists on the Portal, but the iu_register log did not contain "
                f"'AWS upload successful'. Do not retry blindly; review {raw_result['log_path']}."
            )

        result["status"] = "SUBMITTED_AND_PORTAL_MATCHED"
        results.append(result)
        print(f"VERIFIED: {alias} -> {accession}")
        time.sleep(SECONDS_BETWEEN_REAL_UPLOADS)

    output = pd.DataFrame(results)
    output_path = UPLOAD_WORK_DIR / f"{label}_submit_results.tsv"
    output.to_csv(output_path, sep="\t", index=False, lineterminator="\n")
    print(f"Submission results: {output_path}")
    print(output[["alias", "profile", "status", "accession", "portal_url"]].to_string(index=False))
    return output


def reconcile_rows(frame: pd.DataFrame, output_name: str) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for row in frame.to_dict("records"):
        alias = row["aliases"]
        obj = portal_get(alias)
        if not obj:
            rows.append({
                "day": row.get("_day", ""),
                "profile": row["_profile"],
                "alias": alias,
                "expected_file_set": normalize_identifier(row["file_set"]),
                "expected_md5sum": row["md5sum"],
                "accession": "",
                "portal_url": "",
                "portal_file_set": "",
                "portal_md5sum": "",
                "portal_upload_status": "",
                "portal_object_status": "",
                "verification": "NOT_FOUND",
                "issues": "Portal record not found by alias",
                "submitted_file_name": row["submitted_file_name"],
            })
            continue
        issues = record_comparison(row, obj)
        accession = str(obj.get("accession") or "")
        rows.append({
            "day": row.get("_day", ""),
            "profile": row["_profile"],
            "alias": alias,
            "expected_file_set": normalize_identifier(row["file_set"]),
            "expected_md5sum": row["md5sum"],
            "accession": accession,
            "portal_url": portal_url(accession),
            "portal_file_set": normalize_identifier(obj.get("file_set")),
            "portal_md5sum": str(obj.get("md5sum") or ""),
            "portal_upload_status": str(obj.get("upload_status") or ""),
            "portal_object_status": str(obj.get("status") or ""),
            "verification": "MATCH" if not issues else "MISMATCH",
            "issues": "; ".join(issues),
            "submitted_file_name": row["submitted_file_name"],
        })
    output = pd.DataFrame(rows)
    path = METADATA_DIR / output_name
    output.to_csv(path, index=False, lineterminator="\n")
    print("Reconciliation:", path)
    print(output[[
        "day", "profile", "alias", "accession", "verification",
        "portal_upload_status", "portal_object_status",
    ]].to_string(index=False))
    return output

## 1. Set and test IGVF credentials

Credentials are stored only in the current notebook-kernel environment. They are not written to disk or displayed.

In [4]:
if not os.environ.get("IGVF_API_KEY"):
    os.environ["IGVF_API_KEY"] = getpass.getpass("IGVF_API_KEY: ")
if not os.environ.get("IGVF_SECRET_KEY"):
    os.environ["IGVF_SECRET_KEY"] = getpass.getpass("IGVF_SECRET_KEY: ")

for name in ("IGVF_API_KEY", "IGVF_SECRET_KEY"):
    value = os.environ.get(name, "")
    print(
        f"{name}: {'SET' if value else 'MISSING'}, "
        f"length={len(value)}, whitespace={value != value.strip()}"
    )

# Authenticated read-only check against one active AnalysisSet.
auth_test = portal_get(ANALYSIS_SETS["D0"])
if not auth_test:
    raise RuntimeError("Authentication test did not return the D0 AnalysisSet")
print(
    "Authenticated Portal access confirmed:",
    auth_test.get("accession"),
    auth_test.get("@type"),
)

## 2. Local preflight

This cell checks the ready marker, metadata tables, exact 16-file scope, AnalysisSet assignment, file existence, size, and—by default—MD5 checksums from the final files.

In [5]:
required_paths = [
    READY_MARKER,
    MASTER_MANIFEST,
    H5AD_TSV,
    H5MU_TSV,
    TABULAR_TSV,
    IGVF_PYTHON,
    IU_REGISTER,
]
missing = [str(path) for path in required_paths if not path.is_file()]
if missing:
    raise FileNotFoundError("Required files are missing:\n" + "\n".join(missing))

print(READY_MARKER.read_text(encoding="utf-8"))

manifest = pd.read_csv(MASTER_MANIFEST, sep="\t", dtype=str, keep_default_na=False)
if len(manifest) != 16:
    raise AssertionError(f"Expected 16 rows in the master manifest, found {len(manifest)}")
if not manifest["validation_status"].eq("PASS").all():
    raise RuntimeError("At least one master-manifest row is not locally validated PASS")
if manifest["alias"].duplicated().any():
    raise RuntimeError("Duplicate aliases found in the master manifest")
if manifest["submitted_file_name"].duplicated().any():
    raise RuntimeError("Duplicate file paths found in the master manifest")

alias_to_day = dict(zip(manifest["alias"], manifest["day"]))

h5ad = load_metadata(H5AD_TSV, "matrix_file", "h5ad")
h5mu = load_metadata(H5MU_TSV, "matrix_file", "h5mu")
tabular = load_metadata(TABULAR_TSV, "tabular_file", "tabular")

if (len(h5ad), len(h5mu), len(tabular)) != (4, 4, 8):
    raise AssertionError(
        f"Expected 4 h5ad, 4 h5mu, and 8 tabular rows; found "
        f"{len(h5ad)}, {len(h5mu)}, and {len(tabular)}"
    )

all_rows = pd.concat([h5ad, h5mu, tabular], ignore_index=True)
all_rows["_day"] = all_rows["aliases"].map(alias_to_day)
if all_rows["_day"].isna().any():
    bad = all_rows.loc[all_rows["_day"].isna(), "aliases"].tolist()
    raise RuntimeError(f"Metadata aliases absent from master manifest: {bad}")

if all_rows["aliases"].duplicated().any():
    raise RuntimeError("Duplicate aliases found across metadata TSVs")
if all_rows["submitted_file_name"].duplicated().any():
    raise RuntimeError("Duplicate submitted_file_name values found across metadata TSVs")

local_rows = []
for index, row in enumerate(all_rows.to_dict("records"), start=1):
    path = Path(row["submitted_file_name"])
    day = row["_day"]
    expected_set = ANALYSIS_SETS[day]
    if normalize_identifier(row["file_set"]) != expected_set:
        raise RuntimeError(
            f"Wrong AnalysisSet for {row['aliases']}: "
            f"expected {expected_set}, found {row['file_set']}"
        )
    if not path.is_file():
        raise FileNotFoundError(f"Missing final upload file: {path}")
    if path.stat().st_size <= 0:
        raise RuntimeError(f"Empty final upload file: {path}")

    observed_md5 = md5sum(path) if VERIFY_MD5_BEFORE_UPLOAD else row["md5sum"]
    md5_match = observed_md5 == row["md5sum"]
    print(
        f"[{index:02d}/16] {day} {row['_profile']} "
        f"{path.name}: size={path.stat().st_size:,}, md5_match={md5_match}"
    )
    local_rows.append({
        "day": day,
        "profile": row["_profile"],
        "stage": row["_stage"],
        "alias": row["aliases"],
        "file_set": expected_set,
        "submitted_file_name": str(path),
        "size_bytes": path.stat().st_size,
        "expected_md5sum": row["md5sum"],
        "observed_md5sum": observed_md5,
        "md5_match": md5_match,
    })

local_preflight = pd.DataFrame(local_rows)
local_preflight_path = UPLOAD_WORK_DIR / "local_preflight_16.tsv"
local_preflight.to_csv(
    local_preflight_path,
    sep="\t",
    index=False,
    lineterminator="\n",
)
if not local_preflight["md5_match"].all():
    raise RuntimeError(f"MD5 mismatch detected. Review {local_preflight_path}")

print("\nLocal preflight passed for all 16 files")
print("Report:", local_preflight_path)
print(all_rows.groupby(["_day", "_stage"]).size().to_string())

## 3. Read-only Portal preflight

This checks the four AnalysisSets, two AnalysisStepVersion identifiers, the reference file, and whether any of the 16 aliases already exist. Existing records are accepted only when their MD5 and AnalysisSet match the planned metadata.

In [6]:
# Verify core parent/dependency records without querying every raw SequenceFile.
core_identifiers = set(all_rows["file_set"].map(normalize_identifier))
core_identifiers.update(all_rows["analysis_step_version"].map(normalize_identifier))
for value in all_rows.get("reference_files", pd.Series(dtype=str)):
    core_identifiers.update(normalize_identifier(item) for item in split_identifiers(value))
core_identifiers.discard("")

core_checks = []
for identifier in sorted(core_identifiers):
    obj = portal_get(identifier)
    core_checks.append({
        "identifier": identifier,
        "found": bool(obj),
        "accession": str(obj.get("accession") or "") if obj else "",
        "type": ",".join(obj.get("@type", [])) if obj and isinstance(obj.get("@type"), list) else str(obj.get("@type") or "") if obj else "",
        "status": str(obj.get("status") or "") if obj else "",
    })
    print(identifier, "FOUND" if obj else "NOT FOUND")

core_check_df = pd.DataFrame(core_checks)
core_check_path = UPLOAD_WORK_DIR / "core_dependency_preflight.tsv"
core_check_df.to_csv(core_check_path, sep="\t", index=False, lineterminator="\n")
if not core_check_df["found"].all():
    raise RuntimeError(f"Missing core Portal dependency. Review {core_check_path}")

existing_rows = []
for row in all_rows.to_dict("records"):
    obj = portal_get(row["aliases"])
    issues = record_comparison(row, obj) if obj else []
    existing_rows.append({
        "day": row["_day"],
        "stage": row["_stage"],
        "profile": row["_profile"],
        "alias": row["aliases"],
        "exists": bool(obj),
        "accession": str(obj.get("accession") or "") if obj else "",
        "portal_url": portal_url(str(obj.get("accession") or "")) if obj else "",
        "verification": "MATCH" if obj and not issues else "MISMATCH" if obj else "NOT_FOUND",
        "issues": "; ".join(issues),
    })

existing_preflight = pd.DataFrame(existing_rows)
existing_preflight_path = UPLOAD_WORK_DIR / "existing_alias_preflight.tsv"
existing_preflight.to_csv(
    existing_preflight_path,
    sep="\t",
    index=False,
    lineterminator="\n",
)
if existing_preflight["verification"].eq("MISMATCH").any():
    raise RuntimeError(
        "At least one alias already exists with mismatched metadata. "
        f"Review {existing_preflight_path}; do not POST or PATCH automatically."
    )

print("\nExisting-alias preflight:")
print(existing_preflight.groupby(["stage", "verification"]).size().to_string())
print("Report:", existing_preflight_path)

## 4. Dry-run the four h5ad MatrixFiles

This cell never performs a real submission. It skips any h5ad alias already present with matching metadata.

In [ ]:
h5ad_dryrun_results = dry_run_rows(h5ad, "01_h5ad")
if not h5ad_dryrun_results["status"].isin({
    "DRY_RUN_PASS", "ALREADY_EXISTS_MATCH"
}).all():
    raise RuntimeError("The h5ad dry-run stage did not pass completely")
print("\nH5AD dry-run stage passed")

## 5. Real submission of the four h5ad MatrixFiles

Before running this cell, review the h5ad dry-run logs. Then edit the configuration cell near the top and set:

```python
H5AD_REAL_CONFIRMATION = "UPLOAD_4_H5AD_FILES_TO_IGVF_PROD"
```

Restarting the kernel is not required; rerun the configuration cell after editing it. This cell submits one row at a time and reconciles each alias immediately.

In [ ]:
if H5AD_REAL_CONFIRMATION != EXPECTED_H5AD_TOKEN:
    raise RuntimeError(
        "Real h5ad submission is blocked. Review the dry runs, then set "
        f"H5AD_REAL_CONFIRMATION exactly to {EXPECTED_H5AD_TOKEN!r}."
    )

h5ad_submit_results = submit_rows(h5ad, "01_h5ad")
print("\nH5AD real-submission stage completed")

## 6. Verify all four h5ad records before dependent dry runs

The h5mu and tabular metadata use the h5ad aliases in `derived_from`. This gate requires all four h5ad records to exist and match their planned MD5 and AnalysisSet.

In [ ]:
h5ad_reconciliation = reconcile_rows(h5ad, "h5ad_portal_reconciliation.csv")
if len(h5ad_reconciliation) != 4:
    raise AssertionError("Expected four h5ad reconciliation rows")
if not h5ad_reconciliation["verification"].eq("MATCH").all():
    raise RuntimeError(
        "Not all h5ad records are present and matching. Do not proceed to dependent files."
    )
if h5ad_reconciliation["accession"].nunique() != 4:
    raise RuntimeError("The four h5ad aliases did not resolve to four unique accessions")
print("All four h5ad dependencies are present and matched")

## 7. Dry-run the four h5mu and eight tabular files

This cell checks that every `derived_from` h5ad alias is resolvable, then performs one-row dry runs for the 12 dependent files.

In [ ]:
dependent = pd.concat([h5mu, tabular], ignore_index=True)

for row in dependent.to_dict("records"):
    for identifier in split_identifiers(row.get("derived_from", "")):
        obj = portal_get(identifier)
        if not obj:
            raise RuntimeError(
                f"Missing derived_from dependency {identifier} for {row['aliases']}"
            )

# Preserve the intended order: h5mu first, then guide quantifications, then QC.
dependent_dryrun_results = dry_run_rows(dependent, "02_dependent")
if not dependent_dryrun_results["status"].isin({
    "DRY_RUN_PASS", "ALREADY_EXISTS_MATCH"
}).all():
    raise RuntimeError("The dependent-file dry-run stage did not pass completely")
print("\nDependent-file dry-run stage passed")

## 8. Real submission of the 12 dependent files

Before running this cell, review all dependent-file dry-run logs. Then edit the configuration cell near the top and set:

```python
DEPENDENT_REAL_CONFIRMATION = "UPLOAD_12_DEPENDENT_FILES_TO_IGVF_PROD"
```

The cell submits the four h5mu records first and then the eight tabular records, one row at a time.

In [ ]:
if DEPENDENT_REAL_CONFIRMATION != EXPECTED_DEPENDENT_TOKEN:
    raise RuntimeError(
        "Real dependent-file submission is blocked. Review the dry runs, then set "
        f"DEPENDENT_REAL_CONFIRMATION exactly to {EXPECTED_DEPENDENT_TOKEN!r}."
    )

dependent_submit_results = submit_rows(dependent, "02_dependent")
print("\nDependent-file real-submission stage completed")

## 9. Final Portal reconciliation and accession export

This cell is read-only and can be rerun later to refresh `upload_status` and object `status`. It verifies all 16 aliases against the expected MD5 and timepoint AnalysisSet, then writes a CSV containing every accession and Portal link.

A successful reconciliation confirms record creation and metadata matching. It does not imply that asynchronous DACC/checkfiles validation has already finished.

In [ ]:
final_reconciliation = reconcile_rows(
    all_rows,
    "igvf_analysis_file_accessions_16.csv",
)

found = int(final_reconciliation["accession"].astype(bool).sum())
matched = int(final_reconciliation["verification"].eq("MATCH").sum())
unique_accessions = int(
    final_reconciliation.loc[
        final_reconciliation["accession"].astype(bool), "accession"
    ].nunique()
)

summary_lines = [
    "CC-Perturb-seq IGVF analysis-file submission reconciliation",
    f"checked_utc: {now_utc()}",
    "expected_records: 16",
    f"found_records: {found}",
    f"metadata_matches: {matched}",
    f"unique_accessions: {unique_accessions}",
    f"accession_csv: {METADATA_DIR / 'igvf_analysis_file_accessions_16.csv'}",
    "",
    "Portal object-status counts:",
]
for key, value in final_reconciliation["portal_object_status"].value_counts(dropna=False).items():
    summary_lines.append(f"  {key or '<blank>'}: {value}")
summary_lines.append("")
summary_lines.append("Portal upload-status counts:")
for key, value in final_reconciliation["portal_upload_status"].value_counts(dropna=False).items():
    summary_lines.append(f"  {key or '<blank>'}: {value}")

summary_path = METADATA_DIR / "igvf_analysis_file_upload_summary.txt"
summary_path.write_text("\n".join(summary_lines) + "\n", encoding="utf-8")
print("\n".join(summary_lines))

if found == 16 and matched == 16 and unique_accessions == 16:
    completion_marker = OUTPUT_ROOT / "IGVF_UPLOAD_COMPLETE_16_FILES.txt"
    completion_marker.write_text(
        "\n".join([
            "CC-Perturb-seq IGVF analysis-file records: COMPLETE AND MATCHED",
            f"checked_utc: {now_utc()}",
            "records_found: 16",
            "metadata_matches: 16",
            "unique_accessions: 16",
            f"accession_csv: {METADATA_DIR / 'igvf_analysis_file_accessions_16.csv'}",
            "Note: downstream DACC/checkfiles validation may still be pending.",
            "",
        ]),
        encoding="utf-8",
    )
    print("\nCompletion marker:", completion_marker)
else:
    raise RuntimeError(
        "Final reconciliation is incomplete. Review "
        f"{METADATA_DIR / 'igvf_analysis_file_accessions_16.csv'}"
    )

## Principal outputs

```text
${OUTPUT_ROOT}/metadata/igvf_analysis_file_accessions_16.csv
${OUTPUT_ROOT}/metadata/igvf_analysis_file_upload_summary.txt
${OUTPUT_ROOT}/IGVF_UPLOAD_COMPLETE_16_FILES.txt
${OUTPUT_ROOT}/_upload_work/local_preflight_16.tsv
${OUTPUT_ROOT}/_upload_work/core_dependency_preflight.tsv
${OUTPUT_ROOT}/_upload_work/existing_alias_preflight.tsv
${OUTPUT_ROOT}/_upload_work/01_h5ad_dryrun_results.tsv
${OUTPUT_ROOT}/_upload_work/01_h5ad_submit_results.tsv
${OUTPUT_ROOT}/_upload_work/02_dependent_dryrun_results.tsv
${OUTPUT_ROOT}/_upload_work/02_dependent_submit_results.tsv
${OUTPUT_ROOT}/logs/igvf_utils_analysis_upload/*.log
```

The final accession CSV is the single file to share when you need the accession ID and Portal URL for every new analysis file.